### A2 - AI-Generated Image Detection Using a CNN
#### Problem Definition
This project investigates whether a convolutional neural network can classify images as REAL or AI-GENERATED using the CIFAKE dataset.

The system takes a 32 × 32 RGB image as input and produces a binary
classification indicating whether the image belongs to the REAL or
AI-GENERATED class.

### 1. Environment and Experiment Setup
This section imports the required libraries and defines reproducibility
settings for the machine learning experiment.

In [1]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("Seed:", SEED)

PyTorch version: 2.14.0+cpu
Seed: 42


### 2. Dataset

The CIFAKE dataset contains real and AI-generated images organised into
separate training and testing directories.

In [3]:
data_path = Path("../data/CIFAKE")

train_path = data_path / "train"
test_path = data_path / "test"

print("Train path exists:", train_path.exists())
print("Test path exists:", test_path.exists())

Train path exists: True
Test path exists: True


In [4]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    train_path,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    test_path,
    transform=transform
)

print("Class mapping:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

Class mapping: {'FAKE': 0, 'REAL': 1}
Training images: 100000
Testing images: 20000


In [5]:
from collections import Counter

train_counts = Counter(train_dataset.targets)
test_counts = Counter(test_dataset.targets)

print("Training distribution:", train_counts)
print("Testing distribution:", test_counts)

Training distribution: Counter({0: 50000, 1: 50000})
Testing distribution: Counter({0: 10000, 1: 10000})


In [6]:
idx_to_class = {
    index: class_name
    for class_name, index in train_dataset.class_to_idx.items()
}

for index, count in train_counts.items():
    print(f"Train - {idx_to_class[index]}: {count}")

for index, count in test_counts.items():
    print(f"Test  - {idx_to_class[index]}: {count}")

Train - FAKE: 50000
Train - REAL: 50000
Test  - FAKE: 10000
Test  - REAL: 10000


In [7]:
from sklearn.model_selection import train_test_split

all_train_indices = np.arange(len(train_dataset))
all_train_labels = np.array(train_dataset.targets)

train_indices, val_indices = train_test_split(
    all_train_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=all_train_labels
)

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_dataset))

Training samples: 80000
Validation samples: 20000
Test samples: 20000


In [8]:
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)
test_subset = test_dataset

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_subset,
    batch_size=64,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1250
Validation batches: 313
Test batches: 313


In [9]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel minimum:", images.min().item())
print("Pixel maximum:", images.max().item())

Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])
Pixel minimum: 0.0
Pixel maximum: 1.0
